In [1]:
import pandas as pd
import numpy as np
from pathlib import Path 

## Stock data

In [2]:
stock_dir = Path("/Users/yutung/MQF/Machine learning/Project/ml_project/data/processed/stocks_daily/")

files = sorted(stock_dir.glob("stocks_daily_*.parquet"))
print("Found files:", [f.name for f in files])

Found files: ['stocks_daily_2000_2002.parquet', 'stocks_daily_2003_2005.parquet', 'stocks_daily_2006_2008.parquet', 'stocks_daily_2009_2011.parquet', 'stocks_daily_2012_2014.parquet', 'stocks_daily_2015_2017.parquet', 'stocks_daily_2018_2019.parquet', 'stocks_daily_2020_2021.parquet', 'stocks_daily_2022_2023.parquet', 'stocks_daily_2024_2024.parquet']


In [3]:
df_list = []
for f in files:
    df = pd.read_parquet(f)

    # keep only useful columns
    keep = ["PERMNO", "date", "PRC", "RET", "VOL",
            "SHROUT", "BIDLO", "ASKHI", "NUMTRD"]
    df = df[[c for c in keep if c in df.columns]]

    # date -> datetime
    df["date"] = pd.to_datetime(df["date"].astype(str))

    # basic features
    df["log_ret"] = np.log1p(df["RET"].fillna(0))  # for vol
    df["dollar_volume"] = df["PRC"].abs() * df["VOL"]
    df["mktcap"] = df["PRC"].abs() * df["SHROUT"]
    df["turnover"] = df["VOL"] / df["SHROUT"]
    df["mid"] = (df["BIDLO"] + df["ASKHI"]) / 2
    df["bidask"] = (df["ASKHI"] - df["BIDLO"]) / df["mid"]

    df_list.append(df)

stocks_daily = pd.concat(df_list, ignore_index=True)

# Monthly aggregation
monthly_stock = (
    stocks_daily.set_index("date")
    .groupby("PERMNO")
    .resample("M")
    .agg({
        "RET":      lambda x: (1 + x.fillna(0)).prod() - 1,  # monthly return
        "log_ret":  "std",                                   # monthly vol
        "dollar_volume": "sum",
        "turnover": "mean",
        "mktcap": "last",
        "bidask": "mean",
        "NUMTRD": "sum",
        "PRC": "mean",
    })
    .reset_index()
)

monthly_stock = monthly_stock.rename(columns={
    "RET": "ret",
    "log_ret": "vol",
    "dollar_volume": "dvol",
    "turnover": "turnover",
    "mktcap": "mktcap",
    "bidask": "bidask",
    "NUMTRD": "numtrades",
    "PRC": "price_mean",
})

print(monthly_stock.head())

/var/folders/b4/rh5hpt517nb6_mc9jyr1b1440000gn/T/ipykernel_12467/2454550575.py:29: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  .resample("M")


   PERMNO       date       ret       vol        dvol  turnover    mktcap  \
0   10001 2000-01-31 -0.044119  0.025332  335742.750  0.823122  19906.25   
1   10001 2000-02-29  0.015383  0.012548  181390.625  0.452571  20212.50   
2   10001 2000-03-31 -0.015289  0.026495  581272.875  1.283407  19712.00   
3   10001 2000-04-30  0.011720  0.022529  211757.625  0.562628  19943.00   
4   10001 2000-05-31 -0.023166  0.016333  178489.375  0.407984  19481.00   

     bidask  numtrades  price_mean  
0  0.023530       97.0    7.446875  
1  0.013686       60.0    4.071875  
2  0.018115       98.0    3.774457  
3  0.018016       57.0    4.671052  
4  0.020593       49.0   -0.701705  


## Fundamental Data

In [4]:
dir = "/Users/yutung/MQF/Machine learning/Project/ml_project/data/processed/"
fundamental_data = pd.read_parquet(dir + "fundamentals_quarterly.parquet")
fundamental_data.shape

(914037, 25)

In [5]:
# keep needed columns
cols_keep = [
    "GVKEY", "LPERMNO", "datadate",
    "atq", "ceqq", "cshoq", "ltq", "niq",
    "oibdpq", "revtq", "xintq", "prccq", "cusip",
    "costat", "gsector"
]
fund = fundamental_data[cols_keep].copy()

# parse quarterly date correctly
fund["datadate"] = pd.to_datetime(fund["datadate"].astype(str), format="%Y%m%d")

fund = (
    fund
    .sort_values(["LPERMNO", "datadate"])
    .drop_duplicates(subset=["LPERMNO", "datadate"])
)

# build quarterly features
eps = 1e-9

fund["log_atq"] = np.log(fund["atq"] + eps)
fund["lev_total"] = fund["ltq"] / (fund["atq"] + eps)
fund["equity_ratio"] = fund["ceqq"] / (fund["atq"] + eps)
fund["roa"] = fund["niq"] / (fund["atq"] + eps)
fund["profit_margin"] = fund["niq"] / (fund["revtq"] + eps)
fund["int_coverage"] = fund["oibdpq"] / fund["xintq"].replace(0, np.nan)

fund["mkt_cap"] = fund["prccq"] * fund["cshoq"]
fund["market_to_book"] = fund["mkt_cap"] / (fund["ceqq"] + eps)

fund[["atq_growth", "revtq_growth", "niq_growth"]] = (
    fund.groupby("LPERMNO")[["atq", "revtq", "niq"]].pct_change()
)

# convert quarterly -> monthly (per firm)
def quarterly_to_monthly(g: pd.DataFrame) -> pd.DataFrame:
    """Convert firm-level quarterly data to month-end frequency with ffill."""
    g = g.set_index("datadate").sort_index()

    # build full month-end index between first and last quarter
    month_idx = pd.date_range(
        start=g.index.min().normalize(),
        end=g.index.max().normalize(),
        freq="M"
    )

    g = g.reindex(month_idx).ffill()  # forward-fill quarterly values
    g.index.name = "date"             # this will be our monthly date
    return g

fund_monthly = (
    fund
    .groupby("LPERMNO", group_keys=False)
    .apply(quarterly_to_monthly)
    .reset_index()             # brings 'date' back as a column
    .rename(columns={"LPERMNO": "PERMNO"})
)

print(fund_monthly.head())

/var/folders/b4/rh5hpt517nb6_mc9jyr1b1440000gn/T/ipykernel_12467/1999809476.py:33: FutureWarning: The default fill_method='ffill' in DataFrameGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fund.groupby("LPERMNO")[["atq", "revtq", "niq"]].pct_change()
/var/folders/b4/rh5hpt517nb6_mc9jyr1b1440000gn/T/ipykernel_12467/1999809476.py:42: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  month_idx = pd.date_range(
/var/folders/b4/rh5hpt517nb6_mc9jyr1b1440000gn/T/ipykernel_12467/1999809476.py:55: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after gro

        date    GVKEY   PERMNO     atq   ceqq  cshoq     ltq    niq  oibdpq  \
0 1990-03-31  12994.0  10001.0  20.245  7.243  1.027  13.002  0.728   1.521   
1 1990-04-30  12994.0  10001.0  20.245  7.243  1.027  13.002  0.728   1.521   
2 1990-05-31  12994.0  10001.0  20.245  7.243  1.027  13.002  0.728   1.521   
3 1990-06-30  12994.0  10001.0  18.881  7.179  1.031  11.702  0.035   0.473   
4 1990-07-31  12994.0  10001.0  18.881  7.179  1.031  11.702  0.035   0.473   

   revtq  ...  lev_total  equity_ratio       roa profit_margin  int_coverage  \
0  8.350  ...   0.642233      0.357767  0.035959      0.087186      7.277512   
1  8.350  ...   0.642233      0.357767  0.035959      0.087186      7.277512   
2  8.350  ...   0.642233      0.357767  0.035959      0.087186      7.277512   
3  4.379  ...   0.619776      0.380224  0.001854      0.007993      2.330049   
4  4.379  ...   0.619776      0.380224  0.001854      0.007993      2.330049   

     mkt_cap  market_to_book  atq_growth  re

## Merge monthly_stock + fund_monthly

In [6]:
# align keys (LPERMNO -> PERMNO, datadate -> date)
fund_monthly = fund_monthly.rename(
    columns={"LPERMNO": "PERMNO", "datadate": "date"}
)

# make sure date columns are datetime
fund_monthly["date"] = pd.to_datetime(fund_monthly["date"])
monthly_stock["date"] = pd.to_datetime(monthly_stock["date"])

# merge on PERMNO + date
merged_1 = monthly_stock.merge(
    fund_monthly,
    on=["PERMNO", "date"],
    how="left"
)

print(merged_1.head())
print(merged_1.shape)

   PERMNO       date       ret       vol        dvol  turnover    mktcap  \
0   10001 2000-01-31 -0.044119  0.025332  335742.750  0.823122  19906.25   
1   10001 2000-02-29  0.015383  0.012548  181390.625  0.452571  20212.50   
2   10001 2000-03-31 -0.015289  0.026495  581272.875  1.283407  19712.00   
3   10001 2000-04-30  0.011720  0.022529  211757.625  0.562628  19943.00   
4   10001 2000-05-31 -0.023166  0.016333  178489.375  0.407984  19481.00   

     bidask  numtrades  price_mean  ...  lev_total  equity_ratio       roa  \
0  0.023530       97.0    7.446875  ...   0.763616      0.236384  0.009081   
1  0.013686       60.0    4.071875  ...   0.763616      0.236384  0.009081   
2  0.018115       98.0    3.774457  ...   0.716914      0.283086  0.028691   
3  0.018016       57.0    4.671052  ...   0.716914      0.283086  0.028691   
4  0.020593       49.0   -0.701705  ...   0.716914      0.283086  0.028691   

   profit_margin  int_coverage    mkt_cap  market_to_book  atq_growth  \
0

## Merge macro data

In [7]:
macro = pd.read_excel(dir + "macro_data.xlsx")
# convert date to datetime
macro["date"] = pd.to_datetime(macro["date"].astype(str))

# sort for safety
macro = macro.sort_values("date")

# ensure merged (stock+fund) also uses datetime for date
merged_1["date"] = pd.to_datetime(merged_1["date"])

# merge macro into merged dataset
merged_2 = merged_1.merge(
    macro,
    on="date",
    how="left"
)

print(merged_2.head())
print("shape:", merged_2.shape)

   PERMNO       date       ret       vol        dvol  turnover    mktcap  \
0   10001 2000-01-31 -0.044119  0.025332  335742.750  0.823122  19906.25   
1   10001 2000-02-29  0.015383  0.012548  181390.625  0.452571  20212.50   
2   10001 2000-03-31 -0.015289  0.026495  581272.875  1.283407  19712.00   
3   10001 2000-04-30  0.011720  0.022529  211757.625  0.562628  19943.00   
4   10001 2000-05-31 -0.023166  0.016333  178489.375  0.407984  19481.00   

     bidask  numtrades  price_mean  ...  market_to_book  atq_growth  \
0  0.023530       97.0    7.446875  ...        1.592727    0.099523   
1  0.013686       60.0    4.071875  ...        1.592727    0.099523   
2  0.018115       98.0    3.774457  ...        1.386412   -0.087863   
3  0.018016       57.0    4.671052  ...        1.386412   -0.087863   
4  0.020593       49.0   -0.701705  ...        1.386412   -0.087863   

   revtq_growth  niq_growth        sp500  ir3m  ir10y        vix        gdp  \
0      0.521947   -1.750751  1394.459

In [8]:
industry = pd.read_excel(dir + "industry_data.xlsx")

# convert date column
industry["date"] = pd.to_datetime(industry["date"].astype(str))

# rename ETF columns to avoid confusion
industry = industry.rename(columns={
    "sector": "sector_etf",
    "price": "etf_price",
    "return": "etf_return"
})

# sector mapping
sector_map = {
    10: "XLE",
    15: "XLB",
    20: "XLI",
    25: "XLY",
    30: "XLP",
    35: "XLV",
    40: "XLF",
    45: "XLK",
    50: "XLC",
    55: "XLU",
    60: "XLRE",
}

# add ETF ticker to merged via gsector
merged_2["sector_etf"] = merged_2["gsector"].map(sector_map)

# merge ETF monthly data on (sector_etf, date)
industry = industry.rename(columns={"sector": "sector_etf"})

merged_3 = merged_2.merge(
    industry,
    on=["sector_etf", "date"],
    how="left"
)

print(merged_3.head())
print("shape:", merged_3.shape)

   PERMNO       date       ret       vol        dvol  turnover    mktcap  \
0   10001 2000-01-31 -0.044119  0.025332  335742.750  0.823122  19906.25   
1   10001 2000-02-29  0.015383  0.012548  181390.625  0.452571  20212.50   
2   10001 2000-03-31 -0.015289  0.026495  581272.875  1.283407  19712.00   
3   10001 2000-04-30  0.011720  0.022529  211757.625  0.562628  19943.00   
4   10001 2000-05-31 -0.023166  0.016333  178489.375  0.407984  19481.00   

     bidask  numtrades  price_mean  ...  niq_growth        sp500  ir3m  ir10y  \
0  0.023530       97.0    7.446875  ...   -1.750751  1394.459961  5.53  6.667   
1  0.013686       60.0    4.071875  ...   -1.750751  1366.420044  5.64  6.409   
2  0.018115       98.0    3.774457  ...    1.882000  1498.579956  5.72  6.023   
3  0.018016       57.0    4.671052  ...    1.882000  1452.430054  5.65  6.214   
4  0.020593       49.0   -0.701705  ...    1.882000  1420.599976  5.49  6.285   

         vix        gdp    cpi  sector_etf  etf_price  e

## Merge bond data

In [9]:
bond_data = pd.read_parquet(dir + "bond_data_processed.parquet")
print(bond_data.head())

        date      cusip company_symbol   tmt   coupon  t_spread    yield  \
0 2002-07-31  000325AA8           AAFM  0.55  0.08875       NaN      NaN   
1 2002-08-31  000325AA8           AAFM  0.47  0.08875    0.0042  0.07731   
2 2002-09-30  000325AA8           AAFM  0.38  0.08875    0.0078  0.07933   
3 2002-10-31  000325AA8           AAFM  0.30  0.08875       NaN  0.07708   
4 2002-11-30  000325AA8           AAFM  0.21  0.08875    0.0025  0.04793   

    ret_eom  rating_A  rating_AA  ...  rating_BB  rating_BBB  rating_C  \
0       NaN       NaN        NaN  ...        NaN         NaN       NaN   
1  0.005162       0.0        0.0  ...        0.0         0.0       0.0   
2  0.007239       0.0        0.0  ...        0.0         0.0       0.0   
3  0.014820       0.0        0.0  ...        0.0         0.0       0.0   
4 -0.000199       0.0        0.0  ...        0.0         0.0       0.0   

   rating_CC  rating_CCC  rating_D  upgrade  downgrade    gs3m  term_spread  
0        NaN        

In [ ]:
def normalize_cusip6(s):
    s = s.astype(str).str.strip().str.upper()
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.str.pad(6, fillchar='0')
    return s.str[:6]

bond_data['issuer6'] = normalize_cusip6(bond_data['cusip'])

# find stock CUSIP root
if 'cusip' in merged_3.columns:
    merged_3['issuer6'] = normalize_cusip6(merged_3['cusip'])
elif 'cusip_firm' in merged_3.columns:
    merged_3['issuer6'] = normalize_cusip6(merged_3['cusip_firm'])
else:
    print("merged_3 has no cusip column!")

bond_data['month'] = bond_data['date'].dt.to_period('M')
merged_3['month']  = merged_3['date'].dt.to_period('M')



In [14]:
# find common issuer
bond_issuers = set(bond_data["issuer6"].dropna().unique())
merged_issuers = set(merged_3["issuer6"].dropna().unique())
common_issuers = bond_issuers & merged_issuers

print("bond issuers :", len(bond_issuers))
print("merged issuers:", len(merged_issuers))
print("valid issuers:", len(common_issuers))


bond issuers : 5699
merged issuers: 16280
valid issuers: 1486


In [23]:
# select bond_data
bond_filtered = bond_data[bond_data["issuer6"].isin(common_issuers)].copy()

print("before:", bond_data.shape)
print("after :", bond_filtered.shape)

# shift features in merged_3 since bond features are lagged by 1 month
cols_to_shift = [

    # stock
    "ret", "vol", "dvol", "turnover",
    "mktcap", "bidask", "numtrades", "price_mean",

    # fundamentals
    "atq","ceqq","cshoq","ltq","niq",
    "oibdpq","revtq","xintq","prccq",

    # derived
    "log_atq","lev_total","equity_ratio","roa",
    "profit_margin","int_coverage","mkt_cap",
    "market_to_book",
    "atq_growth","revtq_growth","niq_growth",

    # macro
    "sp500","ir3m","ir10y","vix","gdp","cpi",

    # industry
    "etf_price","etf_return",
]

merged_3_lag = (
    merged_3.sort_values(["issuer6","date"])
    .groupby("issuer6", group_keys=False)
    .apply(lambda g: g.assign(**{c: g[c].shift(1) for c in cols_to_shift}))
)


print("merged_3：")
print(merged_3.loc[merged_3["issuer6"] == list(common_issuers)[0],
                   ["issuer6", "date"] + cols_to_shift[:3]].head(5))

print("\n merged_3_lag：")
print(merged_3_lag.loc[merged_3_lag["issuer6"] == list(common_issuers)[0],
                       ["issuer6", "date"] + cols_to_shift[:3]].head(5))


before: (3567979, 24)
after : (871712, 24)
merged_3：
        issuer6       date       ret       vol        dvol
1783380  539439 2001-12-31  0.072466  0.011951  88778968.0
1783381  539439 2002-01-31 -0.034229  0.011889  81881152.0
1783382  539439 2002-02-28 -0.066322  0.022935  51419096.0
1783383  539439 2002-03-31  0.059755  0.020993  26994760.0
1783384  539439 2002-04-30  0.117112  0.010987  23646880.0

 merged_3_lag：
        issuer6       date       ret       vol        dvol
1783380  539439 2001-12-31       NaN       NaN         NaN
1783381  539439 2002-01-31  0.072466  0.011951  88778968.0
1783382  539439 2002-02-28 -0.034229  0.011889  81881152.0
1783383  539439 2002-03-31 -0.066322  0.022935  51419096.0
1783384  539439 2002-04-30  0.059755  0.020993  26994760.0


/var/folders/b4/rh5hpt517nb6_mc9jyr1b1440000gn/T/ipykernel_12467/1312734986.py:34: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.assign(**{c: g[c].shift(1) for c in cols_to_shift}))


In [24]:
merged_all = bond_filtered.merge(
    merged_3_lag,
    on=["issuer6", "date"],
    how="left"
)

print(merged_all.head())

        date    cusip_x company_symbol   tmt  coupon  t_spread    yield  \
0 2002-07-31  000361AB1            AIR  1.23  0.0725       NaN      NaN   
1 2002-08-31  000361AB1            AIR  1.14  0.0725       NaN  0.04827   
2 2002-09-30  000361AB1            AIR  1.06  0.0725       NaN  0.04386   
3 2002-10-31  000361AB1            AIR  0.97  0.0725       NaN  0.04122   
4 2002-11-30  000361AB1            AIR  0.89  0.0725       NaN  0.03873   

    ret_eom  rating_A  rating_AA  ...       sp500   ir3m  ir10y        vix  \
0       NaN       NaN        NaN  ...  989.820007  1.659  4.824  25.400000   
1  0.008709       0.0        0.0  ...  911.619995  1.665  4.465  32.029999   
2  0.006141       0.0        0.0  ...  916.070007  1.645  4.137  32.639999   
3  0.005690       0.0        0.0  ...  815.280029  1.527  3.607  39.689999   
4  0.001961       0.0        0.0  ...  885.760010  1.417  3.911  31.139999   

         gdp    cpi  sector_etf  etf_price  etf_return  month_y  
0  14460.848  

In [25]:
dir = "/Users/yutung/MQF/Machine learning/Project/ml_project/data/"
merged_all.to_parquet(dir + "merged_all.parquet", index=False)